### Notebook collecting commands/scripts which are run after fits/scans complete

In [12]:
# imports
# autoreload magic:
%load_ext autoreload
%autoreload 2
import glob
import copy
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
from scipy.stats import norm
%matplotlib inline
import sys

from NNMFit.utilities import load_pickle
from NNMFit.utilities import ScanHandler

unblind_script_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/"
sys.path.append(unblind_script_path)
from freefit_parameter_config import FreefitParamConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


run scan analysis: calculate MC Histogram, Data Histogram, and Saturated LLH of the best fit

In [13]:
fit_path = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor"

saturated_path   = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/notebooks/unblind/step2_hese_flavor/saturated"

config_path      = "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/"
config_input_path = f"{config_path}/analysis_configs/data/hese/step2_hese_flavor/input_pars_only"
config_plot_path  = f"{config_path}/analysis_configs/data/hese/step2_hese_flavor/plotting"
config_pse_path   = f"{config_path}/analysis_configs/pse/hese/step2_hese_flavor"

os.system(f"mkdir -p {saturated_path}")
os.system(f"mkdir -p {config_input_path}")
os.system(f"mkdir -p {config_plot_path}")
os.system(f"mkdir -p {config_pse_path}")

0

In [9]:
scans = {
    # first scan, 50 freefits
    "all_param_2sigma_1D_10steps": 
        f"{fit_path}/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps",

    # # angle and scaling comparison
    # "scaling": 
    #     f"{fit_path}/data_fits_angle/scaling/",
    # "angle": 
    #     f"{fit_path}/data_fits_angle/angle/",

}

In [5]:
for name, scan_dir in scans.items():
    # run do_scan_analysis.py with scan_dir as arg:
    print(f"Processing scan directory: {scan_dir}")
    os.system(f"python {unblind_script_path}/do_scan_analysis.py {scan_dir} --sat_llh_save_dir {saturated_path}/{name}")

Processing scan directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps
Auto-Detecting the best fit from among the freefits in the scan path.
Initializing on server cobalt-14. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only


/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'astro_nue_ratio', 'astro_nutau_ratio',
       'barr_h', 'barr_w', 'barr_y', 'barr_z', 'conv_norm', 'delta_gamma',
       'dom_eff', 'fit_success', 'gamma_astro', 'ice_abs', 'ice_crystal',
       'ice_holep0', 'ice_holep1', 'ice_scat', 'llh', 'muongun_norm',
       'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  188.1160509993102
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps
This is the index of the best fit: 42, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps/Freefit_42.pickle
found the minimum llh value:  188.1160509993102
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps
This is the index of the best fit: 42, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps/Freefit_42.pickle
Data Histogram

Collect the bestfit parameter values in a config file (for later pseudoexperiments etc.)

In [14]:
# find the best-fit for each scan and write a configuration file using the
# bestfit params as input params:
freefit_config_names = {}
min_llh_dict = {}
scan_llh = {}
for name, scan_path in scans.items():
    print(scan_path)
    freefit_config_hdl = FreefitParamConfig(scan_path=scan_path)
    print(freefit_config_hdl.get_name())
    freefit_config_hdl.write(custom_name=f"{config_input_path}/{name}.yaml")
    llh = freefit_config_hdl.get_min_llh_freefit(freefit_config_hdl.scan_hdl)
    min_llh_dict[freefit_config_hdl.get_name()] = llh
    scan_llh[name] = llh
    freefit_config_names[os.path.basename(os.path.dirname(scan_path))
                        ] = freefit_config_hdl.get_name()


/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps
Initializing on server cobalt-14. 
Using config directory: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only


/home/pfuerst/software/icecube/NNMFit/NNMFit/utilities/result_handlers.py:201: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['CR_grad', 'astro_norm', 'astro_nue_ratio', 'astro_nutau_ratio',
       'barr_h', 'barr_w', 'barr_y', 'barr_z', 'conv_norm', 'delta_gamma',
       'dom_eff', 'fit_success', 'gamma_astro', 'ice_abs', 'ice_crystal',
       'ice_holep0', 'ice_holep1', 'ice_scat', 'llh', 'muongun_norm',
       'prompt_norm'],
      dtype='object')]

  df.to_hdf(self.scan_result_file, key="scans")


found the minimum llh value:  188.1160509993102
Found 50 freefit files in /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps
This is the index of the best fit: 42, the name is /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/dag_scans/flavor_globalfit/hese/unblind/step2_hese_flavor/data_fits_parameter_scan/nbestfit20_all_param_2sigma_1D_10steps/Freefit_42.pickle
/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit/analysis_configs/input_pars_only/Input_params_data_fits_parameter_scan_nbestfit20_all_param_2sigma_1D_10steps_Freefit_42.yaml
Writing input params to /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit//analysis_configs/data/hese/step2_hese_flavor/input_pars_only/all_param_2sigma_1D_10steps.yaml
found the minimum llh value:  188.1160509993102


In [16]:
# For each scan, create a subfolder with full data configs (separate + combined detectors)
# and component-only variants (Astro / Conv / Muon / flavor).
import yaml
import copy

DETECTOR_CONFIGS_SEPARATE = [
    "IC86_pass2_SnowStorm_FTP_HESE_Cascades",
    "IC86_pass2_SnowStorm_FTP_HESE_DoubleCascades",
    "IC86_pass2_SnowStorm_FTP_HESE_Tracks",
]
DETECTOR_CONFIGS_COMBINED = ["IC86_pass2_SnowStorm_FTP_HESE_Combined"]


def flavor_ratios_to_angles(fr0, fr1, fr2):
    norm = fr0 + fr1 + fr2
    fr0_n, fr1_n, fr2_n = fr0 / norm, fr1 / norm, fr2 / norm
    sphi4 = (1.0 - fr2_n) ** 2
    sphi2 = 1.0 - fr2_n
    if sphi2 == 0.0:
        c2psi = 0.0
    else:
        x = np.sqrt(fr0_n / sphi2)
        if x > 1.0:
            x = 1.0
        c2psi = np.cos(2.0 * np.arccos(x))
    return sphi4, c2psi   # a = sphi4,  b = c2psi


def write_data_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: data\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


def write_pse_config(params, detector_configs, output_path, overrides=None):
    p = copy.deepcopy(params)
    if overrides:
        p.update(overrides)
    det_cfg_str = "[" + ",".join(detector_configs) + "]"
    with open(output_path, "w") as f:
        f.write("analysis_type: pseudoexp\n")
        f.write("pseudoexp_type: poisson\n")
        f.write("llh: SAYLLH\n")
        f.write(f"detector_configs: {det_cfg_str}\n")
        f.write("input_params:\n")
        for key, val in p.items():
            f.write(f"  {key}: {val}\n")


for scan_name, scan_path in scans.items():
    input_yaml_path = f"{config_input_path}/{scan_name}.yaml"
    with open(input_yaml_path) as f:
        base_data = yaml.safe_load(f)
    params = base_data["input_params"]

    astro_norm        = params["astro_norm"]
    astro_nue_ratio   = params["astro_nue_ratio"]
    astro_nutau_ratio = params["astro_nutau_ratio"]

    # Convert fitted ratios to flavor fractions, then to (a, b) angle parameters
    f_e   = astro_nue_ratio   / (1 + astro_nue_ratio + astro_nutau_ratio)
    f_mu  = 1.0               / (1 + astro_nue_ratio + astro_nutau_ratio)
    f_tau = astro_nutau_ratio / (1 + astro_nue_ratio + astro_nutau_ratio)
    a, b  = flavor_ratios_to_angles(f_e, f_mu, f_tau)

    # Total normalization = NuMu (×1) + NuE (×nue_ratio) + NuTau (×nutau_ratio)
    total_astro_norm = astro_norm * (1 + astro_nue_ratio + astro_nutau_ratio)

    params["total_astro_norm"] = total_astro_norm
    params["gamma"] = params["gamma_astro"]
    params["a"] = a
    params["b"] = b
    print(f"[{scan_name}] astro_nue_ratio={astro_nue_ratio:.4f}  astro_nutau_ratio={astro_nutau_ratio:.4f}")
    print(f"             f_e={f_e:.4f}  f_mu={f_mu:.4f}  f_tau={f_tau:.4f}")
    print(f"             a={a:.6f}  b={b:.6f}  total_astro_norm={total_astro_norm:.6f}")

    # Astro        — astro_SPL_no_inel.yaml     (ratio params, best-fit composition)
    # Astro_3flavor — astro_SPL_3flavor_no_inel.yaml (angle params, best-fit composition)
    # Flavor components — pure-flavor a/b angles, total_astro_norm scaled by each flavor's ratio:
    #   NuE: (a=1, b=1),  NuMu: (a=1, b=-1),  NuTau: (a=0, b=0)
    #   total_astro_norm: astro_norm × nue_ratio / × 1 / × nutau_ratio
    COMPONENT_OVERRIDES = {
        "Astro":         {"conv_norm": 0, "muongun_norm": 0},
        "Astro_3flavor": {"conv_norm": 0, "muongun_norm": 0, "total_astro_norm": total_astro_norm,                "a": a,   "b": b},
        "Astro_NuE":     {"conv_norm": 0, "muongun_norm": 0, "total_astro_norm": astro_norm * astro_nue_ratio,   "a": 1.0, "b": 1.0},
        "Astro_NuMu":    {"conv_norm": 0, "muongun_norm": 0, "total_astro_norm": astro_norm * 1.0,               "a": 1.0, "b": -1.0},
        "Astro_NuTau":   {"conv_norm": 0, "muongun_norm": 0, "total_astro_norm": astro_norm * astro_nutau_ratio, "a": 0.0, "b": 0.0},
        "Conv":          {"astro_norm": 0, "muongun_norm": 0},
        "Muon":          {"astro_norm": 0, "conv_norm": 0},
    }

    # pse configs
    write_pse_config(params, DETECTOR_CONFIGS_SEPARATE, f"{config_pse_path}/{scan_name}.yaml")
    print(f"Written: {config_pse_path}/{scan_name}.yaml")

    # data configs for plotting
    scan_dir = f"{config_plot_path}/{scan_name}"
    os.makedirs(scan_dir, exist_ok=True)

    variants = [
        (scan_name,               DETECTOR_CONFIGS_SEPARATE),
        (f"{scan_name}_combined", DETECTOR_CONFIGS_COMBINED),
    ]

    for stem, det_configs in variants:
        write_data_config(params, det_configs, f"{scan_dir}/{stem}.yaml")
        print(f"Written: {scan_dir}/{stem}.yaml")

        for component, overrides in COMPONENT_OVERRIDES.items():
            write_data_config(params, det_configs, f"{scan_dir}/{stem}_{component}.yaml", overrides=overrides)
            print(f"Written: {scan_dir}/{stem}_{component}.yaml")

[all_param_2sigma_1D_10steps] astro_nue_ratio=0.5016  astro_nutau_ratio=0.0000
             f_e=0.3341  f_mu=0.6659  f_tau=0.0000
             a=1.000000  b=-0.331895  total_astro_norm=8.441986
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit//analysis_configs/pse/hese/step2_hese_flavor/all_param_2sigma_1D_10steps.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit//analysis_configs/data/hese/step2_hese_flavor/plotting/all_param_2sigma_1D_10steps/all_param_2sigma_1D_10steps.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit//analysis_configs/data/hese/step2_hese_flavor/plotting/all_param_2sigma_1D_10steps/all_param_2sigma_1D_10steps_Astro.yaml
Written: /data/user/tvaneede/GlobalFit/reco_processing/NNMFit/configs/flavor_globalfit//analysis_configs/data/hese/step2_hese_flavor/plotting/all_param_2sigma_1D_10steps/all_param_2sigma_1D_10steps_Astro_3flavor.yaml
Written: /da